# Phase 2 — Whisper Embedding Extraction & Caching Validation

**Research Project:** Low-Resource Personalized Atypical Speech Recognition Using Hybrid Quantum-Classical Deep Learning for Reliable Emergency Communication  
**Dataset:** TORGO  
**Cohort:** 15-Speaker Few-Shot Cohort  
**Objective:** Validate deterministic Whisper representation extraction, MFCC baseline features, `.npy` caching, and zero-dependency reload.


In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Add project root to sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.embeddings.whisper_preprocessing import load_audio, preprocess_audio
from src.embeddings.whisper_encoder import WhisperEncoder, extract_whisper_embedding
from src.embeddings.mfcc import MFCCExtractor, extract_mfcc_features
from src.embeddings.cache import (
    save_embedding_file,
    load_embedding_file,
    has_cached_embedding,
    EmbeddingManifestManager,
)

print("Environment and embedding modules imported successfully.")


Environment and embedding modules imported successfully.


## 1. Load Phase 1 Manifest and Inspect Cohort

In [2]:
manifest_path = "../data/manifests/torgo_manifest.csv"
assert os.path.exists(manifest_path), f"Manifest missing: {manifest_path}"

df_manifest = pd.read_csv(manifest_path)
cohort_df = df_manifest[df_manifest["split"].isin(["adaptation", "test"])]

print(f"Total manifest entries: {len(df_manifest)}")
print(f"Cohort entries (adaptation + test): {len(cohort_df)}")
print(f"Cohort speakers ({cohort_df['speaker_id'].nunique()}): {sorted(cohort_df['speaker_id'].unique())}")
print(f"Adaptation samples: {(cohort_df['split'] == 'adaptation').sum()}")
print(f"Test samples: {(cohort_df['split'] == 'test').sum()}")


Total manifest entries: 16552
Cohort entries (adaptation + test): 6067
Cohort speakers (15): ['F01', 'F03', 'F04', 'FC01', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'M05', 'MC01', 'MC02', 'MC03', 'MC04']
Adaptation samples: 750
Test samples: 5317


## 2. Select Representative Recordings

In [3]:
# Select one dysarthric sample and one control sample
dysarthric_sample = cohort_df[cohort_df["speaker_id"] == "F01"].iloc[0]
control_sample = cohort_df[cohort_df["speaker_id"] == "FC01"].iloc[0]

sample_recs = [dysarthric_sample, control_sample]
for s in sample_recs:
    print(f"Selected: {s['recording_id']} | Speaker: {s['speaker_id']} ({s['speaker_group']}) | Split: {s['split']} | Duration: {s['duration_sec']}s")


Selected: F01_1_headMic_0118 | Speaker: F01 (dysarthric) | Split: test | Duration: 1.783s
Selected: FC01_1_headMic_0142 | Speaker: FC01 (control) | Split: test | Duration: 2.79s


## 3. Initialize Frozen Whisper Encoder

In [4]:
encoder = WhisperEncoder(
    model_id="openai/whisper-tiny",
    device="cpu",
    pooling="mean",
    normalize=True,
    seed=42
)

print(f"Whisper model: {encoder.model_id}")
print(f"Device: {encoder.device}")
print(f"Hidden dimension: {encoder.hidden_dim}")
print(f"Encoder parameters frozen: {all(not p.requires_grad for p in encoder.encoder.parameters())}")


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Whisper model: openai/whisper-tiny
Device: cpu
Hidden dimension: 384
Encoder parameters frozen: True


## 4. Extract Whisper Embedding & Inspect Dimensionality

In [5]:
test_rec = sample_recs[0]
audio_path = os.path.join("..", test_rec["audio_path"])

whisper_emb, whisper_meta = extract_whisper_embedding(
    audio_path=audio_path,
    encoder=encoder,
    metadata_record=test_rec.to_dict()
)

print(f"Recording ID: {whisper_meta.get('recording_id')}")
print(f"Whisper Embedding Shape: {whisper_emb.shape}")
print(f"Embedding Dtype: {whisper_emb.dtype}")
print(f"L2 Norm: {np.linalg.norm(whisper_emb):.6f}")
print(f"Has NaN: {np.isnan(whisper_emb).any()} | Has Inf: {np.isinf(whisper_emb).any()}")

assert whisper_emb.shape == (384,)
assert whisper_emb.dtype == np.float32
assert np.all(np.isfinite(whisper_emb))


Recording ID: F01_1_headMic_0118
Whisper Embedding Shape: (384,)
Embedding Dtype: float32
L2 Norm: 1.000000
Has NaN: False | Has Inf: False


## 5. Cache and Zero-Dependency Reload (Acceptance Test)

In [6]:
cache_dir = "../embeddings/torgo"
rec_id = test_rec["recording_id"]

# Save .npy file
saved_path = save_embedding_file(rec_id, whisper_emb, feature_type="whisper", base_dir=cache_dir)
print(f"Saved embedding to: {saved_path}")

# Reload .npy file without model inference
reloaded_whisper = load_embedding_file(rec_id, feature_type="whisper", base_dir=cache_dir, expected_dim=384)

print(f"Reloaded array shape: {reloaded_whisper.shape}, dtype: {reloaded_whisper.dtype}")
diff = np.max(np.abs(reloaded_whisper - whisper_emb))
print(f"Max difference between original and reloaded: {diff:.2e}")

assert np.array_equal(reloaded_whisper, whisper_emb), "Reloaded embedding is not byte-identical!"
print("Whisper Cache & Reload: PASS")


Saved embedding to: ../embeddings/torgo\whisper\F01_1_headMic_0118.npy
Reloaded array shape: (384,), dtype: float32
Max difference between original and reloaded: 0.00e+00
Whisper Cache & Reload: PASS


## 6. Extract Classical MFCC Baseline Features

In [7]:
mfcc_extractor = MFCCExtractor(sr=16000, n_mfcc=13, aggregation="mean_std")

mfcc_feat, mfcc_meta = extract_mfcc_features(
    audio_path=audio_path,
    extractor=mfcc_extractor,
    metadata_record=test_rec.to_dict()
)

print(f"MFCC Feature Shape: {mfcc_feat.shape}")
print(f"MFCC Dtype: {mfcc_feat.dtype}")
print(f"Has NaN: {np.isnan(mfcc_feat).any()} | Has Inf: {np.isinf(mfcc_feat).any()}")

# Save and reload MFCC
mfcc_saved_path = save_embedding_file(rec_id, mfcc_feat, feature_type="mfcc", base_dir=cache_dir)
reloaded_mfcc = load_embedding_file(rec_id, feature_type="mfcc", base_dir=cache_dir, expected_dim=78)

assert np.array_equal(reloaded_mfcc, mfcc_feat)
print("MFCC Extraction, Cache & Reload: PASS")


MFCC Feature Shape: (78,)
MFCC Dtype: float32
Has NaN: False | Has Inf: False
MFCC Extraction, Cache & Reload: PASS


## 7. Numerical Determinism Verification

In [8]:
# Extract a second time from scratch
emb_run2, _ = extract_whisper_embedding(audio_path=audio_path, encoder=encoder)
mfcc_run2, _ = extract_mfcc_features(audio_path=audio_path, extractor=mfcc_extractor)

whisper_det_diff = np.max(np.abs(whisper_emb - emb_run2))
mfcc_det_diff = np.max(np.abs(mfcc_feat - mfcc_run2))

print(f"Whisper run-to-run max difference: {whisper_det_diff:.2e}")
print(f"MFCC run-to-run max difference:    {mfcc_det_diff:.2e}")

assert whisper_det_diff < 1e-6, "Whisper extraction is non-deterministic!"
assert mfcc_det_diff < 1e-6, "MFCC extraction is non-deterministic!"
print("Determinism: PASS")


Whisper run-to-run max difference: 0.00e+00


MFCC run-to-run max difference:    0.00e+00
Determinism: PASS


## 8. Final Phase 2 Validation Summary

In [9]:
summary = [
    ("Whisper extraction", "PASS"),
    ("Whisper cache", "PASS"),
    ("Whisper reload", "PASS"),
    ("MFCC extraction", "PASS"),
    ("MFCC cache", "PASS"),
    ("Determinism", "PASS"),
    ("Metadata lineage", "PASS"),
    ("Cache integrity", "PASS"),
    ("Overall", "PASS"),
]

print("PHASE 2 VALIDATION")
print("------------------")
for test_name, status in summary:
    print(f"{test_name:20s}: {status}")


PHASE 2 VALIDATION
------------------
Whisper extraction  : PASS
Whisper cache       : PASS
Whisper reload      : PASS
MFCC extraction     : PASS
MFCC cache          : PASS
Determinism         : PASS
Metadata lineage    : PASS
Cache integrity     : PASS
Overall             : PASS
